# 第 11 章实践与延伸阅读


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 客观题

1. （填空）Gated-KV 的三段结构：______（SIMT）、______（SIMT）、______（Cube/Vector）。
2. （选择）教学版门控主线用方案 A（Host 随机分数）的原因：
   A. 性能更好<br/>B. 先隔离"选择后执行"的误差面，门控来源后置<br/>C. 无法实现方案 B
3. （判断）有限输入、非空可见集合及相同归一化约定下，跳过未选块与加 -inf 掩码数学上等价；实际速度仍需测量。（对/错）

In [ ]:
!cat ./answer/11.06_objective_answer.txt

## 2. 编程实践一（简单）：golden 脚本与选择推演

**任务**：① 实现 11.02 练习 3 的 torch 参考脚本（g → 稳定排序前 k → gather → torch SDPA），固定 seed，为三档 ratio 各生成一组 golden 并保存；② 推演：S=4096、KV_BLOCK=128、ratio=12.5% 时选中块数、QK^T 计算量相对全量的比例。后续两个实践都用此脚本验收。


In [ ]:
# 在此完成实践一（golden 生成 + 推演记录）
pass

## 3. 编程实践二（中等）：改造三步走（步骤 1——布尔跳过版）

**任务**：在第 10 章教学版 SDPA 上实施 11.04 步骤 1：`sel_mask`（Host 传入 UINT8/INT32 的 0/1 表 [num_blocks]）控制 K 块循环跳过。验收：功能对照 gather 版 golden；记录 ratio=25% 时相对全量版的加速比。


In [ ]:
# 在此完成实践二（0/1 掩码跳过版）
pass

## 4. 编程实践三（困难）：教学版完整实现（步骤 2+3）

**任务**：把选择段上 NPU（SIMT 确定性排名 + 按 rank 写索引，复用第 9 章实践模块），并把循环改为 sel_idx 查表遍历。验收：选择一致性 + 数值 + 三档 ratio 耗时曲线，填 11.05 记录表。


In [ ]:
# 在此完成实践三（完整实现与验收）
pass

In [ ]:
!cat ./answer/11.06_practice_answer.txt

## 5. 延伸阅读：对照 gym 的稀疏 MLA 与 Indexer

【gym 依赖标注：仅源码延伸阅读，课程运行不导入 gym】本轮核对的目录是 **`ops/pypto_pro/deepseek_v4_1/`**，不是旧稿的 `ops/pro/deepseek_v4_1/`。以下映射依据 README、Kernel 与测试契约整理，目录名本身不足以说明完整模型。

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">算子目录</th><th style="text-align:left;">实际功能</th><th style="text-align:left;">与课程的比较点</th></tr>
<tr><td>indexer_prologue_qw / indexer_prologue_k</td><td>Q/W 或 K 投影、RoPE、量化、缓存前处理</td><td>选择前的输入准备，不等于 top-k 本身</td></tr>
<tr><td>quant_lightning_indexer</td><td>多 head 相关性分数、逐步 BF16 舍入、精确 TopK 与可选候选块</td><td>对照选择算法；该实现允许同分集合不同，而本课程要求块号破平，不能照搬验收规则</td></tr>
<tr><td>mixed_quant_sparse_flash_mla</td><td>消费外部稀疏索引，融合反量化、QK、带 sink 的 online softmax、PV，支持分片归并</td><td>对照注意力消费段；它不在内部替代本章选择器，KV 表示与本课程独立 K/V 也不同</td></tr>
<tr><td>engram_gate</td><td>RMS/加权点积得到 sigmoid gate，执行 x+gate·value 的残差更新</td><td>可复习第 3 章向量融合；它不是 KV 块 TopK 选择器</td></tr>
</table>

阅读每个目录的 README、`op_kernel/*.py` 和 `tests/assets/*golden*`，分别记录输入、舍入点、选择规则、同步及输出。部分算子 README 使用 CANN 9.3.0，不能据此宣称课程目标 CANN 9.2.0 环境已经兼容。`ops/pro/experimental/flex_attention/` 目前仍是另一条实际存在的参考路径，其完整 mask 契约超出本课程主线。


### 完整参考实现与验收

提供步骤 2+3 的完整阶段实现、同语义 Golden、三档 ratio 与两种 mask 验收；`--full` 扩展 S/H/B，`--benchmark` 输出时间。只依赖课程第 10 章参考，不导入 gym。0/1 跳过版和同 Kernel 发布融合仍作为增量练习。

[参考源码](./src/gated_kv_reference.py) 可在 950 上从章节目录执行 `python src/gated_kv_reference.py`。下列单元仅展示源码；设备编译、精度与性能均以实际执行为准。


In [ ]:
!cat ./src/gated_kv_reference.py


## 6. 课程结语

回顾 11 章路径：硬件地图 → 编程模型 → SIMD 轨道 → SIMT 轨道 → 协同 → 调优 → 双实战。完成这些实践后，可以逐项检查自己是否掌握了读规格、设计计算图、选择协同结构、编写 Kernel、验收精度和闭环调优。下一步建议：挑 gym 仓一个实验性算子（如 `ops/pro/experimental/flex_attention`）做精读，或为 pypto-gym 贡献你的第一个算子——课程结束，语料开始。